<a href="https://colab.research.google.com/github/23f1000190/Important-colab-or-notebooks/blob/main/CRNN_for_milestone_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import librosa
import wandb
import random
import kagglehub
import tqdm
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [2]:
!pip install wandb -q

In [3]:
!pip install librosa

In [4]:
# import data from kaggle
from google.colab import files
files.upload()

Saving kaggle (1).json to kaggle (1).json


{'kaggle (1).json': b'{"username":"priyanshu23f1000190","key":"4cb91d988321b70f3a30e18c67778a30"}'}

In [5]:
os.environ["KAGGLE_USERNAME"] = "priyanshu23f1000190"
os.environ["KAGGLE_KEY"] = "4cb91d988321b70f3a30e18c67778a30"

In [6]:
path = kagglehub.competition_download("jan-2026-dl-gen-ai-project")
print("Dataset downloaded to:", path)

100%|██████████| 16.1G/16.1G [02:56<00:00, 97.9MB/s]


Extracting files...
Dataset downloaded to: /root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project


In [14]:
SR = 44100
FULL_LENGTH = 30 * SR
SEGMENT = 10 * SR
#N_MELS = 128

#BATCH_SIZE = 16
#EPOCHS = 15
#LR = 1e-4

In [7]:
wandb.login(relogin=True)
wandb.init(
    project= "DL-23f100190-notebook-t12026",
    entity= "23f1000190-dl-genai-project",
    name="CRNN_pipeline",
    config= {
        "model": "CRNN_scratch",
        "lr": 1e-4,
        "batch_size": 16,
        "epochs":15,
        "segment_seconds": 10,
        "mel_bins":128
    }

)


config =wandb.config


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: ERROR Invalid API key: API key must have 40+ characters, has 1.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

...

wandb: ERROR Invalid API key: API key must have 40+ characters, has 1.
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


In [8]:
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_PATH = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems"
NOISE_PATH = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup/ESC-50-master/audio"
TEST_PATH  = "/root/.cache/kagglehub/competitions/jan-2026-dl-gen-ai-project/messy_mashup"

In [9]:
# ----------------------------------------------------------
# GENRE MAPPING
# ----------------------------------------------------------

genres = sorted(os.listdir(TRAIN_PATH))

genre_to_label = {g:i for i,g in enumerate(genres)}
label_to_genre = {i:g for g,i in genre_to_label.items()}

print("Genre mapping:", genre_to_label)

Genre mapping: {'blues': 0, 'classical': 1, 'country': 2, 'disco': 3, 'hiphop': 4, 'jazz': 5, 'metal': 6, 'pop': 7, 'reggae': 8, 'rock': 9}


In [10]:
# ----------------------------------------------------------
# BUILD TRAIN/VAL LIST
# ----------------------------------------------------------

samples = []
genre_to_folders = {i: [] for i in range(len(genres))}

for genre in genres:

    genre_path = os.path.join(TRAIN_PATH, genre)
    label = genre_to_label[genre]

    for folder in os.listdir(genre_path):

        folder_path = os.path.join(genre_path, folder)

        samples.append((folder_path, label))
        genre_to_folders[label].append(folder_path)

random.shuffle(samples)

split = int(0.8 * len(samples))

train_samples = samples[:split]
val_samples   = samples[split:]

print("Train:", len(train_samples))
print("Val:", len(val_samples))

Train: 800
Val: 200


In [11]:
# LOAD NOISE FILES
# ----------------------------------------------------------

noise_files = []

for f in os.listdir(NOISE_PATH):
    if f.endswith(".wav"):
        noise_files.append(os.path.join(NOISE_PATH, f))

print("Noise files:", len(noise_files))


Noise files: 2000


In [16]:
# ----------------------------------------------------------
# DATASET
# ----------------------------------------------------------

class GenreDataset(Dataset):

    def __init__(self, samples, genre_to_folders, noise_files=None, augment=False):

        self.samples = samples
        self.genre_to_folders = genre_to_folders
        self.noise_files = noise_files
        self.augment = augment


    def __len__(self):
        return len(self.samples)


    def load_stem(self, folder, stem):

        path = os.path.join(folder, f"{stem}.wav")

        waveform, _ = librosa.load(path, sr=SR)

        if len(waveform) > FULL_LENGTH:
            waveform = waveform[:FULL_LENGTH]
        else:
            waveform = np.pad(waveform, (0, FULL_LENGTH - len(waveform)))

        return waveform


    def __getitem__(self, idx):

        folder_path, label = self.samples[idx]

        bass = self.load_stem(folder_path, "bass")
        drums = self.load_stem(folder_path, "drums")
        vocals = self.load_stem(folder_path, "vocals")
        other = self.load_stem(folder_path, "other")

        waveform = bass + drums + vocals + other
        waveform = np.clip(waveform, -1, 1)

        waveform = waveform[:SEGMENT]

        mel = librosa.feature.melspectrogram(
            y=waveform,
            sr=SR,
            n_mels=config['mel_bins']
        )

        log_mel = librosa.power_to_db(mel)

        log_mel = (log_mel - log_mel.mean())/(log_mel.std()+1e-6)

        log_mel = np.expand_dims(log_mel,0)

        return torch.tensor(log_mel,dtype=torch.float32), torch.tensor(label)

In [17]:
# ----------------------------------------------------------
# DATALOADERS
# ----------------------------------------------------------

train_dataset = GenreDataset(train_samples, genre_to_folders, noise_files, augment=True)
val_dataset   = GenreDataset(val_samples, genre_to_folders, augment=False)

train_loader = DataLoader(train_dataset,batch_size=config['batch_size'],shuffle=True)
val_loader   = DataLoader(val_dataset,batch_size=config['batch_size'])

In [18]:
# ----------------------------------------------------------
# CRNN MODEL
# ----------------------------------------------------------

class CRNN(nn.Module):

    def __init__(self,num_classes=10):

        super(CRNN,self).__init__()

        # CNN block 1
        self.conv1 = nn.Conv2d(1,32,3,padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.pool1 = nn.MaxPool2d(2,2)

        # CNN block 2
        self.conv2 = nn.Conv2d(32,64,3,padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.pool2 = nn.MaxPool2d(2,2)

        # BiLSTM
        self.lstm = nn.LSTM(
            input_size=64*32,
            hidden_size=64,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.fc = nn.Linear(128,num_classes)


    def forward(self,x):

        x = self.pool1(torch.relu(self.bn1(self.conv1(x))))
        x = self.pool2(torch.relu(self.bn2(self.conv2(x))))

        B,C,F,T = x.size()

        x = x.permute(0,3,1,2)
        x = x.contiguous().view(B,T,C*F)

        x,_ = self.lstm(x)

        x = torch.max(x,dim=1)[0]

        x = self.fc(x)

        return x


In [20]:
# ----------------------------------------------------------
# MODEL SETUP
# ----------------------------------------------------------

model = CRNN(num_classes=len(genres)).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),lr=config['lr'])
criterion = nn.CrossEntropyLoss()


In [23]:
# ----------------------------------------------------------
# TRAINING
# ----------------------------------------------------------

best_acc = 0

for epoch in range(config['epochs']):
    model.train()
    train_loss = 0
    train_preds=[]
    train_targets=[]

    model.train()

    train_preds=[]
    train_targets=[]

    for x,y in (train_loader):

        x=x.to(DEVICE)
        y=y.to(DEVICE)

        optimizer.zero_grad()

        logits=model(x)

        loss=criterion(logits,y)

        loss.backward()

        optimizer.step()

        train_preds+=logits.argmax(1).cpu().tolist()
        train_targets+=y.cpu().tolist()


    train_acc=accuracy_score(train_targets,train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro")
    model.eval()

    val_preds=[]
    val_targets=[]

    with torch.no_grad():

        for x,y in val_loader:

            x=x.to(DEVICE)

            logits=model(x)

            val_preds+=logits.argmax(1).cpu().tolist()
            val_targets+=y.tolist()


    val_acc=accuracy_score(val_targets,val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    })




    print("Epoch",epoch+1,"Val Acc",val_acc)

    if val_acc>best_acc:

        best_acc=val_acc
        torch.save(model.state_dict(),"best_model.pth")


print("Training complete")

Epoch 1 Val Acc 0.285
Epoch 2 Val Acc 0.465
Epoch 3 Val Acc 0.485
Epoch 4 Val Acc 0.51
Epoch 5 Val Acc 0.495
Epoch 6 Val Acc 0.535
Epoch 7 Val Acc 0.53
Epoch 8 Val Acc 0.555
Epoch 9 Val Acc 0.595
Epoch 10 Val Acc 0.625
Epoch 11 Val Acc 0.615
Epoch 12 Val Acc 0.585
Epoch 13 Val Acc 0.635
Epoch 14 Val Acc 0.675
Epoch 15 Val Acc 0.67
Training complete


In [24]:
wandb.login(relogin=True)
wandb.init(
    project= "DL-23f100190-notebook-t12026",
    entity= "23f1000190-dl-genai-project",
    name="CRNN_pipeline2",
    config= {
        "model": "CRNN_scratch",
        "lr": 1e-3,
        "batch_size": 16,
        "epochs":10,
        "segment_seconds": 10,
        "mel_bins":128
    }

)


config =wandb.config

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


In [25]:

model = CRNN(num_classes=len(genres)).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(),lr=config['lr'])
criterion = nn.CrossEntropyLoss()


In [26]:

train_dataset = GenreDataset(train_samples, genre_to_folders, noise_files, augment=True)
val_dataset   = GenreDataset(val_samples, genre_to_folders, augment=False)

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
val_loader   = DataLoader(val_dataset,batch_size=32)

In [ ]:
# ----------------------------------------------------------
# TRAINING
# ----------------------------------------------------------

best_acc = 0

for epoch in range(config['epochs']):
    model.train()
    train_loss = 0
    train_preds=[]
    train_targets=[]

    model.train()

    train_preds=[]
    train_targets=[]

    for x,y in (train_loader):

        x=x.to(DEVICE)
        y=y.to(DEVICE)

        optimizer.zero_grad()

        logits=model(x)

        loss=criterion(logits,y)

        loss.backward()

        optimizer.step()

        train_preds+=logits.argmax(1).cpu().tolist()
        train_targets+=y.cpu().tolist()


    train_acc=accuracy_score(train_targets,train_preds)
    train_f1 = f1_score(train_targets, train_preds, average="macro")
    model.eval()

    val_preds=[]
    val_targets=[]

    with torch.no_grad():

        for x,y in val_loader:

            x=x.to(DEVICE)

            logits=model(x)

            val_preds+=logits.argmax(1).cpu().tolist()
            val_targets+=y.tolist()


    val_acc=accuracy_score(val_targets,val_preds)
    val_f1 = f1_score(val_targets, val_preds, average="macro")

    wandb.log({
        "epoch": epoch,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    })




    print("Epoch",epoch+1,"Val Acc",val_acc)

    if val_acc>best_acc:

        best_acc=val_acc
        torch.save(model.state_dict(),"crnn_model.pth")


print("Training complete")

Epoch 1 Val Acc 0.29
